In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Imports

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

from typing import List

# Parameters

In [ ]:
# Time boundaries (microseconds)
START_DATE = dt.datetime(2025, 12, 1, 0, 0, 0).timestamp() * 1_000_000
END_DATE   = dt.datetime(2025, 12, 2, 0, 0, 0).timestamp() * 1_000_000

SYMBOLS = ['btcusdt', 'ethusdt']
BASE_PATH = '/content/drive/MyDrive/CMF_HFT/liquidation_task'

# Data Downloading

In [ ]:
def load_sym(sym):
    bin_trades = pl.scan_parquet(f'{BASE_PATH}/data/binance_trades/perp_{sym}.parquet') \
                   .filter(pl.col('timestamp').is_between(START_DATE, END_DATE))
    bin_bbo = pl.scan_parquet(f'{BASE_PATH}/data/binance_booktickers/perp_{sym}.parquet') \
                .filter(pl.col('timestamp').is_between(START_DATE, END_DATE))
    bin_liq = pl.scan_parquet(f'{BASE_PATH}/data/binance_liquidations/perp_{sym}.parquet') \
                .filter(pl.col('timestamp').is_between(START_DATE, END_DATE))
    byb_liq = pl.scan_parquet(f'{BASE_PATH}/data/bybit_liquidations/{sym}.parquet') \
                .filter(pl.col('timestamp').is_between(START_DATE, END_DATE))
    return {
        'bin_trades': bin_trades,
        'bin_bbo': bin_bbo,
        'bin_liq': bin_liq,
        'byb_liq': byb_liq
    }

data = {sym: load_sym(sym) for sym in SYMBOLS}

data['btcusdt']['bin_trades'].schema

# Quick preview
for sym in SYMBOLS:
    print(f'\n=== {sym} ===')
    for name, lf in data[sym].items():
        print(f'{name}: {lf.select(pl.len()).collect().item()} rows')

# Task 3.1 Official Scoring, Sanity Checks, and Basic Validation

## Markout and Score Calculation

In [ ]:
def add_mid(bbo_lf: pl.LazyFrame) -> pl.LazyFrame:
    return bbo_lf.with_columns(
        ((pl.col('bid_price') + pl.col('ask_price')) / 2).alias('mid')
    )

def add_notional(trades_lf: pl.LazyFrame) -> pl.LazyFrame:
    return trades_lf.with_columns(
        (pl.col('price') * pl.col('amount')).alias('notional')
    )

def compute_markout(
    trades_lf: pl.LazyFrame,
    bbo_lf: pl.LazyFrame,
    horizon_s: List[int] = [30, 120, 300],
) -> pl.LazyFrame:
    """
    Computes maker PnL (in bps) for each trade at multiple horizons.
    Formula: pnl = -sign * (mid(t+h) - price) / price * 10000 + 0.5
    sign = +1 for taker-buy (maker-sell), -1 for taker-sell (maker-buy).
    For each horizon h, a column pnl_{h} is added.
    If BBO at time t+h is missing, pnl_{h} = null.
    """
    # Prepare BBO
    bbo_mid = add_mid(bbo_lf).select('timestamp', 'mid').sort('timestamp')

    # Base LazyFrame with trades
    df = (
        trades_lf
        .pipe(add_notional)
        .sort('timestamp')
        .with_columns(
            sign=pl.when(pl.col('side') == 'buy')
                   .then(pl.lit(1))
                   .otherwise(pl.lit(-1))
        )
    )

    # For each horizon, do a separate asof join and save mid to a separate column
    for h in horizon_s:
        horizon_us = h * 1_000_000
        ts_future_col = f'ts_future_{h}'
        mid_col = f'mid_{h}'
        df = df.with_columns(
            (pl.col('timestamp') + horizon_us).alias(ts_future_col)
        )
        df = df.join_asof(
            bbo_mid,
            left_on=ts_future_col,
            right_on='timestamp',
            strategy='backward'
        )
        df = df.rename({'mid': mid_col})
        df = df.drop('timestamp_right')

    # Compute pnl for each horizon
    for h in horizon_s:
        mid_col = f'mid_{h}'
        pnl_col = f'pnl_{h}'
        df = df.with_columns(
            (
                -1
                * pl.col('sign')
                * (pl.col(mid_col) - pl.col('price'))
                / pl.col('price')
                * 10_000
                + 0.5
            ).alias(pnl_col)
        )

    # Add weight (common for all horizons)
    df = df.with_columns(
        weight=pl.col('notional').clip(upper_bound=100_000)
    )

    return df

In [ ]:
def pnl_all(sym: str) -> None:
    """
    Computes and prints PnL_all for three horizons (30, 120, 300 seconds).
    PnL_all(t) = sum_i w_i * pnl_i(t) / sum_i w_i
    where w_i = min(notional_i, 100_000).
    Trades with missing BBO mid at horizon are excluded.
    """
    if sym not in data:
        raise ValueError(f"Symbol {sym} not found in data")
    trades = data[sym]['bin_trades']
    bbo = data[sym]['bin_bbo']

    # compute markout with weights and pnls
    lazy_result = compute_markout(trades, bbo)
    df = lazy_result.collect()

    for h in [30, 120, 300]:
        pnl_col = f'pnl_{h}'
        # отфильтровываем сделки, для которых pnl отсутствует
        valid = df.filter(pl.col(pnl_col).is_not_null())
        if valid.height == 0:
            print(f"PnL_all_{sym} {h}s: нет валидных сделок")
            continue

        total_weight = valid['weight'].sum()
        weighted_sum = (valid['weight'] * valid[pnl_col]).sum()
        pnl_all_val = weighted_sum / total_weight

        print(f"PnL_all_{sym} {h}s: {pnl_all_val:.2f} bps (средневзвешенное, n={valid.height})")
    print()


# Вызов для всех символов
for sym in SYMBOLS:
    pnl_all(sym)

## Quality Metrics

**Main quality groups of filters:**

### 1. Effectiveness

- **Score(τ)** – the primary target metric. For each horizon τ ∈ {30, 120, 300} s.  
`Score = PnL_kept - PnL_all` – the higher, the better.
- Additionally, we compute **PnL_filtered** (on filtered trades) and **PnL_all** – to understand whether the filter cuts off unprofitable trades. The lower, the better.

### 2. Stability

- **Mean / Standard deviation** of hourly PnL_kept values.  
Gives an idea of strategy volatility. The higher, the better.

- **Max Drawdown** – characterises robustness. The lower, the better.

### 3. Turnover Constraint

- **KeptTurnoverPerDay** – average daily clipped turnover of the kept trades.  
  - Should be ≥ 500,000 USD. All else being equal, the higher the better, so that in case of adverse market conditions we can reduce turnover and still stay within the limit.

- **Average time** between 2 kept trades. All else being equal, the smaller the better, for the same reason as above.


**to be continued... 🔜🔜🔜**